# 06 · The Attention Mechanism

**Time:** about 45–60 minutes · **Prerequisites:** `04` (or `05`)

## Learning objectives
1. Implement scaled dot-product attention yourself, and check it against
   MLX's fused kernel.
2. Show **experimentally** why scores are divided by $\sqrt{d_k}$.
3. Watch attention patterns get **learned**: train query/key projections on a
   small task and compare the heatmap before and after.
4. Apply **causal masks** and know the MLX mask conventions.

$$\text{Attention}(Q, K, V) = \text{softmax}\!\left(\frac{QK^\top}{\sqrt{d_k}}\right)V$$

**The motivation.** An LSTM squeezes everything it has read into one hidden
vector. Attention lets each position look back at every position directly:
* a **query** $q$: what this position is looking for;
* **keys** $k_j$: what each position offers to be matched against;
* **values** $v_j$: what each position hands over if it is attended to.

The output for a query is a weighted average of the values, weighted by how
well the query matches each key.

In [ ]:
import numpy as np
import mlx.core as mx
import mlx.nn as nn
import mlx.optimizers as optim
import matplotlib.pyplot as plt
import seaborn as sns

from mlx_nlp_utils import print_device_info, sanity_check

sns.set_style('whitegrid')
print_device_info()

## Part 1: Implement attention

Four lines, each matching one piece of the formula. Shapes are
`(batch, sequence, features)`.

In [ ]:
def attention(Q, K, V, mask=None):
    """Scaled dot-product attention returning (output, weights).

    mask: boolean array, True where attention is ALLOWED (MLX convention).
    """
    d_k = Q.shape[-1]
    scores = Q @ K.swapaxes(-1, -2) / np.sqrt(d_k)         # (batch, queries, keys)
    if mask is not None:
        scores = mx.where(mask, scores, -float('inf'))      # blocked → probability 0
    weights = mx.softmax(scores, axis=-1)                   # each row sums to 1
    return weights @ V, weights                             # (batch, queries, features)

mx.random.seed(42)
Q = mx.random.normal((1, 5, 8))
K = mx.random.normal((1, 5, 8))
V = mx.random.normal((1, 5, 8))
output, weights = attention(Q, K, V)
print("output", output.shape, "| weights", weights.shape, "| row sums", np.array(weights.sum(-1)).round(3))

### Check it against two other implementations

Never trust a hand-written numerical function without a test. We compare
against the reference in `mlx_nlp_utils` and against
`mx.fast.scaled_dot_product_attention`, MLX's fused kernel. The fused kernel
expects a heads axis, `(batch, heads, sequence, features)`, and returns only
the output: it never builds the full weight matrix, which saves memory on
long sequences. That is why notebook 07 uses it, and why we use this explicit
version when we want to *look* at the weights.

In [ ]:
from mlx_nlp_utils import scaled_dot_product_attention as reference_attention

ref_out, ref_w = reference_attention(Q, K, V)
fast_out = mx.fast.scaled_dot_product_attention(Q[:, None], K[:, None], V[:, None], scale=8 ** -0.5)[:, 0]
np.testing.assert_allclose(np.array(output), np.array(ref_out), atol=1e-5)
np.testing.assert_allclose(np.array(output), np.array(fast_out), atol=1e-5)
print("✅ All three implementations agree")

In [ ]:
plt.figure(figsize=(6, 5))
sns.heatmap(np.array(weights[0]), annot=True, fmt='.2f', cmap='viridis', square=True,
            xticklabels=[f"key {i}" for i in range(5)], yticklabels=[f"query {i}" for i in range(5)])
plt.title("Attention weights for random Q and K"); plt.show()

With random queries and keys, the pattern is arbitrary: nothing has been
learned yet. Part 3 trains the projections and shows a pattern that
*means* something.

## Part 2: Why divide by $\sqrt{d_k}$?

If the entries of $q$ and $k$ have mean 0 and variance 1, their dot product
$q \cdot k = \sum_{i=1}^{d_k} q_i k_i$ has variance $d_k$. Larger dimensions
produce larger scores, and softmax of large scores is nearly **one-hot**.
A one-hot softmax has almost zero gradient: the model can't learn to shift
attention.

🤔 **Predict:** without scaling, at $d_k = 512$, what is the typical largest
attention weight among 32 keys?

In [ ]:
def attention_stats(d, scale, n_keys=32, trials=200, seed=0):
    """Mean max-weight and entropy, median gradient size, and score std of softmax(q·k * scale)."""
    mx.random.seed(seed)
    q = mx.random.normal((trials, 1, d))
    k = mx.random.normal((trials, n_keys, d))
    scores = (q @ k.swapaxes(-1, -2))[:, 0, :] * scale            # (trials, keys)
    w = mx.softmax(scores, axis=-1)
    entropy = -mx.sum(w * mx.log(w + 1e-12), axis=-1)
    # How much does the attention distribution respond to a change in scores?
    target = mx.random.normal((trials, n_keys))
    grad = mx.grad(lambda s: mx.sum(mx.softmax(s, axis=-1) * target))(scores)
    grad_norm = np.array(mx.sqrt(mx.sum(grad ** 2, axis=-1)))
    # Median, not mean: a few unsaturated trials would otherwise dominate the average.
    return (float(w.max(axis=-1).mean()), float(entropy.mean()),
            float(np.median(grad_norm)), float(scores.std()))

dims = [4, 16, 64, 256, 512, 1024]
stats = {label: [attention_stats(d, s(d)) for d in dims]
         for label, s in [("unscaled", lambda d: 1.0), ("scaled by 1/√d", lambda d: d ** -0.5)]}

fig, axes = plt.subplots(1, 4, figsize=(18, 3.8))
titles = ['score std', 'largest attention weight', 'entropy (nats)', 'median gradient size']
for label, rows in stats.items():
    rows = np.array(rows)
    for ax, col, title in zip(axes, [3, 0, 1, 2], titles):
        ax.plot(dims, rows[:, col], 'o-', label=label); ax.set_xscale('log', base=2); ax.set_title(title)
axes[2].axhline(np.log(32), color='gray', linestyle=':', label='uniform over 32 keys')
for ax in axes:
    ax.set_xlabel('d_k'); ax.legend(fontsize=8)
axes[3].set_yscale('log')
plt.tight_layout(); plt.show()

for label, rows in stats.items():
    print(f"{label:<16} d=512: max weight {rows[4][0]:.2f}, median gradient size {rows[4][2]:.5f}")
sanity_check(stats["unscaled"][4][2] < stats["scaled by 1/√d"][4][2] / 5,
             "unscaled attention has a much smaller gradient at d=512")

Without scaling, score spread grows like $\sqrt{d_k}$, the largest weight
approaches 1, entropy collapses toward 0, and the gradient shrinks: a
saturated softmax that barely learns. (The gradient panel uses a log scale:
at $d_k = 512$ the typical unscaled gradient is about 75 times smaller, and far smaller beyond.) With scaling, all four curves stay
flat as $d_k$ grows. That is the whole reason for the $\sqrt{d_k}$.

## Part 3: Attention is learned

In a Transformer, $Q$, $K$, and $V$ are not given; they are **linear
projections** of the token representations:
$Q = XW_Q,\; K = XW_K,\; V = XW_V$.
Training adjusts $W_Q$ and $W_K$ until the attention pattern is useful.

A small task to watch this happen: each example is a sequence of 8 distinct
tokens plus one **query token** that appears somewhere in the sequence. The
model must point at the position holding the query token, using only its
attention scores. That requires learning projections under which a
token's query matches its own key.

In [ ]:
VOCAB, SEQ, D = 16, 8, 16

def make_batch(n, rng):
    seqs = np.stack([rng.permutation(VOCAB)[:SEQ] for _ in range(n)])
    target_pos = rng.integers(0, SEQ, size=n)
    queries = seqs[np.arange(n), target_pos]
    return mx.array(seqs), mx.array(queries), mx.array(target_pos)

class PointerAttention(nn.Module):
    """Embeddings + learned query/key projections; returns attention scores over positions."""
    def __init__(self):
        super().__init__()
        self.embed = nn.Embedding(VOCAB, D)
        self.W_q = nn.Linear(D, D, bias=False)
        self.W_k = nn.Linear(D, D, bias=False)

    def __call__(self, seqs, queries):
        q = self.W_q(self.embed(queries))[:, None, :]     # (batch, 1, D)
        k = self.W_k(self.embed(seqs))                     # (batch, SEQ, D)
        return (q @ k.swapaxes(-1, -2))[:, 0, :] / np.sqrt(D)   # scores (batch, SEQ)

rng = np.random.default_rng(0)
mx.random.seed(0)
pointer = PointerAttention()
demo_seqs, demo_q, demo_pos = make_batch(6, np.random.default_rng(123))
before = np.array(mx.softmax(pointer(demo_seqs, demo_q), axis=-1))

loss_and_grad = nn.value_and_grad(
    pointer, lambda m, s, q, p: nn.losses.cross_entropy(m(s, q), p, reduction='mean'))
optimizer = optim.Adam(learning_rate=0.01)
train_steps = 300
accuracy_curve = []
for step in range(train_steps):
    seqs, queries, pos = make_batch(64, rng)
    loss, grads = loss_and_grad(pointer, seqs, queries, pos)
    optimizer.update(pointer, grads)
    mx.eval(pointer.parameters(), optimizer.state)
    if step % 10 == 0:
        accuracy_curve.append(float(mx.mean(mx.argmax(pointer(seqs, queries), axis=-1) == pos)))

after = np.array(mx.softmax(pointer(demo_seqs, demo_q), axis=-1))
fig, axes = plt.subplots(1, 3, figsize=(17, 3.8))
for ax, w, title in [(axes[0], before, 'before training'), (axes[1], after, 'after training')]:
    sns.heatmap(w, ax=ax, cmap='viridis', vmin=0, vmax=1, cbar=False, annot=True, fmt='.1f')
    ax.set_title(f'Attention from query token ({title})'); ax.set_xlabel('position'); ax.set_ylabel('example')
    ax.scatter(np.array(demo_pos) + 0.5, np.arange(6) + 0.5, marker='s', s=250, facecolors='none', edgecolors='red')
axes[2].plot(np.arange(len(accuracy_curve)) * 10, accuracy_curve)
axes[2].axhline(1 / SEQ, color='gray', linestyle=':', label='chance')
axes[2].set_title('pointing accuracy'); axes[2].set_xlabel('step'); axes[2].set_ylim(0, 1.05); axes[2].legend()
plt.tight_layout(); plt.show()
print(f"Final batch accuracy: {accuracy_curve[-1]:.0%} (chance = {1 / SEQ:.0%}); red boxes mark the correct position")
sanity_check(accuracy_curve[-1] > 0.9, "learned attention points at the matching token")

Before training, attention is spread arbitrarily. After a few hundred
steps it concentrates on the matching position. Nobody wrote a rule
"attend to the same token"; gradient descent found projections $W_Q, W_K$
that make it happen. In a language model, the same mechanism learns patterns
such as "attend to the previous token" or "attend to the subject of this
verb", and notebook 07 visualizes some.

## Part 4: Cross-attention (illustration)

In translation, queries come from the sentence being generated (French) and
keys and values from the source sentence (English). Below, $Q$ and $K$ are
**constructed by hand** so each French word's query equals its English
counterpart's key. This shows what a learned alignment looks like; a trained
model has to find it, as in Part 3.

In [ ]:
english = ["I", "love", "machine", "learning"]
french = ["J'", "aime", "apprentissage", "automatique"]
mx.random.seed(42)
directions = mx.random.normal((len(english), 16))
K_en = directions[None]
Q_fr = directions[mx.array([0, 1, 3, 2])][None]   # "apprentissage"↔"learning", "automatique"↔"machine"
_, w_cross = attention(Q_fr, K_en, mx.random.normal((1, 4, 16)))

plt.figure(figsize=(6, 4))
sns.heatmap(np.array(w_cross[0]), annot=True, fmt='.2f', cmap='YlOrRd', xticklabels=english, yticklabels=french)
plt.title("Cross-attention (hand-constructed alignment)"); plt.xlabel("English keys"); plt.ylabel("French queries")
plt.show()

Notice the word-order swap: French puts *apprentissage* (learning) before
*automatique* (machine). Attention handles reordering naturally because any
query can match any key.

## Part 5: Causal masking

A language model predicting token $t$ must not see tokens after $t$. A
**causal mask** blocks those positions. In MLX, a boolean mask uses `True`
for **allowed**; an additive float mask uses `0` for allowed and `-inf` for blocked.

In [ ]:
words = ["The", "cat", "sat", "on", "mat"]
causal_mask = mx.tril(mx.ones((5, 5), dtype=mx.bool_))
print("Causal mask (True = visible):")
print(np.array(causal_mask).astype(int))

mx.random.seed(7)
Qc, Kc, Vc = (mx.random.normal((1, 5, 8)) for _ in range(3))
_, w_full = attention(Qc, Kc, Vc)
out_causal, w_causal = attention(Qc, Kc, Vc, mask=causal_mask)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
for ax, w, title in [(axes[0], w_full, "No mask"), (axes[1], w_causal, "Causal mask")]:
    sns.heatmap(np.array(w[0]), annot=True, fmt='.2f', cmap='viridis', square=True,
                xticklabels=words, yticklabels=words, ax=ax)
    ax.set_title(title); ax.set_xlabel("keys"); ax.set_ylabel("queries")
plt.tight_layout(); plt.show()

fast_causal = mx.fast.scaled_dot_product_attention(Qc[:, None], Kc[:, None], Vc[:, None],
                                                  scale=8 ** -0.5, mask="causal")[:, 0]
np.testing.assert_allclose(np.array(fast_causal), np.array(out_causal), atol=1e-5)
print("✅ Our masked attention matches mx.fast with mask='causal'")

Both panels use the same $Q$, $K$, $V$. With the mask, the upper triangle
is exactly zero, and each row renormalizes over the positions it may see:
"The" can only attend to itself, so its weight is 1.

## Summary
* Attention is a softmax-weighted average of values, weighted by
  query–key similarity, in four lines of code.
* Dividing by $\sqrt{d_k}$ keeps the softmax out of saturation, so
  gradients survive as dimensions grow.
* Useful attention patterns are **learned** through $W_Q$ and $W_K$.
* Causal masks make attention respect time; MLX boolean masks mark *allowed* positions.

## 🧠 Exercises

**1. Temperature in disguise.** Replace `/ np.sqrt(d_k)` with `/ (2 * np.sqrt(d_k))`
in `attention` and rerun the random heatmap. 🤔 Predict the effect.

<details><summary>What to look for</summary>

Dividing scores by a larger number flattens the softmax, exactly like
sampling temperature in notebook 03. The scale is a temperature on attention.
</details>

**2. How small can a head be?** In Part 3, set `D = 2`, then `D = 1`.
🤔 Predict: can the model still point at the right token among 16 possible tokens?

<details><summary>What to look for</summary>

`D = 2` still reaches ~100%: the 16 token embeddings can sit at different
angles around a circle, and a query pointing the same way as its own key
scores highest. `D = 1` fails (around 25% in our runs): with scalar queries and
keys, $q \cdot k$ is largest for whichever key is most extreme, so a middle
token can never win. The geometry of the query/key space limits which
patterns one head can express.
</details>

**3. Untrained keys.** Freeze `W_k` (`pointer.W_k.freeze()` before training)
and retrain. Does it still work?

<details><summary>What to look for</summary>

Usually yes: the query projection and embeddings can adapt to a fixed random
key projection. Attention has redundant parameters, which is one reason
methods like LoRA (notebook 08) can adapt a model by changing only a few of them.
</details>

## ❓ FAQ

**Is attention the same as memory?** It behaves like content-addressable
memory: the query asks "who has X?", matching keys answer, and their values
are returned.

**Next:** `06b_Prompt_Engineering` measures how attention behaves on long
prompts; `07_Build_NanoGPT` assembles attention into a full GPT.